In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_data import prepare_prices


# 01 Stock Data
Download the constituents and daily adjusted prices, then form the chronological 70/30 split here. Run all cells in order. The constituents CSV is a current snapshot, not historical membership: the experiment retains survivorship bias.


## 1. Settings


In [ ]:
CONFIG = ResearchConfig().validate()
cfg = CONFIG
START_DATE = "2016-01-01"
END_DATE = "2026-01-01"  # exclusive
CONSTITUENTS_URL = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
display(pd.Series(cfg.to_dict(), name="Research settings"))


## 2. Download constituents and prices
Yahoo uses hyphens for share classes such as BRK-B. Adjusted Close includes dividend and split adjustments. S&P 500 index observations define the session calendar. Inspect missing downloads before proceeding; no stored price panel is used as a fallback.


In [ ]:
constituents = pd.read_csv(CONSTITUENTS_URL)
tickers = sorted(constituents["Symbol"].str.replace(".", "-", regex=False).unique())
download = yf.download(
    tickers + ["^GSPC"], start=START_DATE, end=END_DATE,
    auto_adjust=False, progress=True, threads=False,
)
close = download["Adj Close"].sort_index()
if "^GSPC" not in close or close["^GSPC"].dropna().empty:
    raise ValueError("S&P 500 calendar download failed. Retry the download cell.")
sessions = close["^GSPC"].dropna().index
prices = close.reindex(index=sessions, columns=tickers)
missing = prices.columns[prices.isna().all()].tolist()
if missing:
    raise ValueError(f"No price history returned for {missing}. Check tickers or retry before proceeding.")
display(prices.head())


## 3. Split 70/30 and check availability
The first 70% of downloaded sessions are formation and the remaining 30% are out of sample. The split is performed by prepare_prices in this cell using cfg.train_fraction=0.7. Asset availability is assessed only in formation; retained assets with missing out-of-sample quotes cause an error rather than being silently dropped.


In [ ]:
train_prices, test_prices, availability = prepare_prices(
    prices, cfg, membership=None, allow_legacy=True
)
display(pd.DataFrame({
    "observations": [len(train_prices), len(test_prices)],
    "start": [train_prices.index.min(), test_prices.index.min()],
    "end": [train_prices.index.max(), test_prices.index.max()],
}, index=["Formation", "Out of sample"]))
display(availability.head(10))


## 4. Save inputs for the following modules


In [ ]:
constituents.to_csv("constituents.csv", index=False)
train_prices.to_parquet("train_prices.parquet")
test_prices.to_parquet("test_prices.parquet")
availability.to_parquet("availability.parquet")
ax = (train_prices.iloc[:, :5] / train_prices.iloc[0, :5]).plot(
    figsize=(10, 4), title="Formation prices normalized to one"
)
ax.set_ylabel("Normalized price")
plt.show()
